# 05: Dashboard & Insights

**Objective**: Present the trend intelligence as an interactive dashboard for
designers and brand decision makers.

### Dashboard Components

1. **Executive Summary**:- Plain language recommendations  
2. **Trend Quadrant Scatter**:- The hero chart (what to over-index on)  
3. **Top-N Recommendations Table**:- Ranked, actionable list  
4. **Style-Color Heatmap**:- Granular sell-through patterns  
5. **Trend Momentum Time Series**:- Google Trends over time  
6. **Category Treemap**:- Hierarchical view by quadrant  


---

## 5.1 Setup

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = "/content/drive/MyDrive/FashionTrendAnalyzer"
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

from src.config import trend_scoring_config, DATA_PROCESSED
from src.data_loader import load_processed
from src.viz import (
    trend_quadrant_scatter,
    style_color_heatmap,
    trend_momentum_timeseries,
    category_treemap,
    recommendations_table,
    methodology_waterfall,
)

print("Dashboard ready.")

Mounted at /content/drive
Dashboard ready.


In [ ]:
# Load all processed data
master = load_processed("trend_intelligence_master")
recs = load_processed("top_recommendations")
sell_through = load_processed("sell_through_scorecard")

# Try loading Google Trends time series
try:
    from src.google_trends import fetch_trends_data
    trends_ts = pd.read_parquet(DATA_PROCESSED / "checkpoint_trends_raw.parquet")
except Exception:
    trends_ts = pd.DataFrame()

print(f"Master: {master.shape} | Recs: {recs.shape} | Sell-through: {sell_through.shape}")

Master: (11, 32) | Recs: (10, 9) | Sell-through: (2634, 17)


---
## 5.2 Executive Summary


In [ ]:
hot_categories = master[master["quadrant"] == "HOT"].sort_values(
    "trend_intelligence_score", ascending=False
)
emerging_categories = master[master["quadrant"] == "EMERGING"].sort_values(
    "trend_intelligence_score", ascending=False
)
fading_categories = master[master["quadrant"] == "FADING"].sort_values(
    "trend_intelligence_score", ascending=False
)

def format_list(df, col="category", max_items=5):
    items = df[col].head(max_items).tolist()
    styles = df.get("dominant_style", pd.Series()).head(max_items).tolist()
    lines = []
    for i, item in enumerate(items):
        style_tag = f" ({styles[i]})" if i < len(styles) and pd.notna(styles[i]) else ""
        lines.append(f"- **{item}**{style_tag}")
    return "\n".join(lines) if lines else "- None identified"

summary = f"""
# Trend Intelligence: Executive Summary

**Analysis Period**: Based on H&M transaction data (2018–2020) and real-time social signals
**Categories Analyzed**: {len(master)}
**Scoring Model**: 5 signal weighted composite (sell-through, sentiment, Google Trends, style buzz, semantic clusters)

---

## OVER-INDEX (HOT: {len(hot_categories)} categories)
Strong sell-through velocity **AND** rising social signals. Increase production allocation.

{format_list(hot_categories)}

## TEST & INVEST (EMERGING:{len(emerging_categories)} categories)
High social buzz but sell-through hasn't caught up yet. Small production runs to test.

{format_list(emerging_categories)}

## MAINTAIN (FADING: {len(fading_categories)} categories)
Still selling well but social interest is declining. Do not increase allocation.

{format_list(fading_categories)}

"""

display(Markdown(summary))


# Trend Intelligence: Executive Summary

**Analysis Period**: Based on H&M transaction data (2018–2020) and real-time social signals
**Categories Analyzed**: 11
**Scoring Model**: 5 signal weighted composite (sell-through, sentiment, Google Trends, style buzz, semantic clusters)

---

## OVER-INDEX (HOT: 2 categories)
Strong sell-through velocity **AND** rising social signals. Increase production allocation.

- **Shorts** (quiet luxury)
- **Prints & Patterns**

## TEST & INVEST (EMERGING:4 categories)
High social buzz but sell-through hasn't caught up yet. Small production runs to test.

- **Other** (oversized comfort)
- **Shoes**
- **Skirts** (quiet luxury)
- **Accessories**

## MAINTAIN (FADING: 4 categories)
Still selling well but social interest is declining. Do not increase allocation.

- **Trousers** (quiet luxury)
- **Knitwear** (oversized comfort)
- **Jackets** (oversized comfort)
- **Tops** (quiet luxury)



---
## 5.3 Trend Quadrant Scatter: The Hero Chart

In [ ]:
if "total_revenue" not in master.columns or master["total_revenue"].isna().all():
    master["total_revenue"] = 1
master["total_revenue"] = master["total_revenue"].fillna(1).clip(lower=1)

fig = trend_quadrant_scatter(master)
fig.update_layout(
    title=dict(text="Trend Intelligence Quadrant: What to Over-Index On", font=dict(size=18)),
    height=650, width=950,
)
fig.show()

---
## 5.4 Top Recommendations Table

In [ ]:
fig = recommendations_table(recs)
fig.update_layout(height=450, width=1100)
fig.show()

---
## 5.5 Style × Color Sell-Through Heatmap

In [ ]:
fig = style_color_heatmap(sell_through)
fig.update_layout(height=550, width=950)
fig.show()

---
## 5.6 Google Trends Momentum Over Time

In [ ]:
fig = trend_momentum_timeseries(trends_ts)
fig.update_layout(height=450, width=950)
fig.show()

---
## 5.7 Category Drill Down Treemap

In [ ]:
fig = category_treemap(master)
fig.update_layout(height=1000, width=950)
fig.show()

---
## 5.8 Methodology Breakdown Score Decomposition

For each of the top-3 categories, show how the five signals contribute
to the final trend intelligence score.

In [ ]:
top3 = master.nlargest(3, "trend_intelligence_score")

weight_map = {
    "Sell-Through": trend_scoring_config.w_sell_through,
    "Sentiment": trend_scoring_config.w_sentiment,
    "Trend Momentum": trend_scoring_config.w_trend_momentum,
    "Style Buzz": trend_scoring_config.w_style_buzz,
    "Cluster Strength": trend_scoring_config.w_cluster_strength,
}

for _, row in top3.iterrows():
    scores = {
        "Sell-Through": row.get("sell_through_composite", 0.5),
        "Sentiment": row.get("sentiment_normalized", 0.5),
        "Trend Momentum": row.get("trend_momentum_normalized", 0.5),
        "Style Buzz": row.get("style_buzz_normalized", 0.5),
        "Cluster Strength": row.get("cluster_strength_normalized", 0.5),
    }
    fig = methodology_waterfall(row["category"], scores, weight_map)
    fig.show()


---
## 5.9 Signal Radar : Multi-Dimensional Category View

In [ ]:
# Radar chart for top 5 categories
radar_cols = [
    "sell_through_composite", "sentiment_normalized",
    "trend_momentum_normalized", "style_buzz_normalized",
    "cluster_strength_normalized",
]
radar_labels = ["Sell-Through", "Sentiment", "Trend Momentum", "Style Buzz", "Cluster Strength"]
available_radar = [c for c in radar_cols if c in master.columns]

top5 = master.nlargest(5, "trend_intelligence_score")

fig = go.Figure()
for _, row in top5.iterrows():
    values = [row.get(c, 0.5) for c in available_radar]
    values.append(values[0])  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels[:len(available_radar)] + [radar_labels[0]],
        name=row["category"],
        fill="toself",
        opacity=0.5,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Signal Radar — Top 5 Categories",
    template="plotly_white",
    height=500, width=700,
    showlegend=True,
)
fig.show()

---
## 5.10 Quadrant Summary Statistics

In [ ]:
quadrant_summary = (
    master
    .groupby("quadrant")
    .agg(
        count=("category", "count"),
        avg_score=("trend_intelligence_score", "mean"),
        avg_sell_through=("sell_through_composite", "mean"),
        avg_social=("social_composite", "mean"),
    )
    .round(3)
    .sort_values("avg_score", ascending=False)
)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Quadrant", "# Categories", "Avg Score", "Avg Sell-Through", "Avg Social"],
        fill_color="#2C3E50",
        font=dict(color="white", size=13),
        align="center",
    ),
    cells=dict(
        values=[
            quadrant_summary.index,
            quadrant_summary["count"],
            quadrant_summary["avg_score"],
            quadrant_summary["avg_sell_through"],
            quadrant_summary["avg_social"],
        ],
        align="center",
        font=dict(size=12),
        height=28,
    ),
)])
fig.update_layout(
    title="Quadrant Summary Statistics",
    height=500, width=700,
)
fig.show()

In [ ]:

!pip install -U kaleido==0.2.1

save_dir = DATA_PROCESSED / "plots"
os.makedirs(save_dir, exist_ok=True)

# save each plot
plots = {
    "01_trend_quadrant": trend_quadrant_scatter(master),
    "02_recommendations": recommendations_table(recs),
    "03_style_heatmap": style_color_heatmap(sell_through),
    "04_trend_momentum": trend_momentum_timeseries(trends_ts),
    "05_category_treemap": category_treemap(master),
}

for name, fig in plots.items():
    path = save_dir / f"{name}.png"
    fig.write_image(str(path), width=1200, height=700, scale=2)
    print(f"Saved {path.name}")

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
